In [1]:
import cv2
import numpy as np

In [ ]:
def zoom_image(image, s, method='bilinear'):
    h, w = image.shape[:2]
    new_h, new_w = int(h * s), int(w * s)
    output = np.zeros((new_h, new_w), dtype=np.uint8)

    # Inverse mapping: for each pixel in output, find coordinate in input
    for i in range(new_h):
        for j in range(new_w):
            # Calculate coordinates in original image
            r = i / s
            c = j / s

            if method == 'nearest':
                # Round to the nearest integer index
                src_r = min(int(round(r)), h - 1)
                src_c = min(int(round(c)), w - 1)
                output[i, j] = image[src_r, src_c]

            elif method == 'bilinear':
                # Find the 4 surrounding pixels
                r1, c1 = int(np.floor(r)), int(np.floor(c))
                r2, c2 = min(r1 + 1, h - 1), min(c1 + 1, w - 1)

                # Fractional distances
                dr, dc = r - r1, c - c1

                # Weighted average of 4 neighbors
                pixel = (1 - dr) * (1 - dc) * image[r1, c1] + \
                        dr * (1 - dc) * image[r2, c1] + \
                        (1 - dr) * dc * image[r1, c2] + \
                        dr * dc * image[r2, c2]
                
                output[i, j] = int(pixel)

    return output

def compute_normalized_ssd(img1, img2):
    # Ensure images are the same size
    if img1.shape != img2.shape:
        img1 = cv2.resize(img1, (img2.shape[1], img2.shape[0]))
    
    # Compute SSD: sum((I1 - I2)^2)
    diff = img1.astype(float) - img2.astype(float)
    ssd = np.sum(diff**2)
    
    # Normalize by number of pixels
    return ssd / (img1.shape[0] * img1.shape[1])

# --- Testing ---
# Load original large image and its small zoomed-out version
original = cv2.imread('a1images/a1q8images/im01.png')
small = cv2.imread('a1images/a1q8images/im01small.png')
print(f"Original loaded: {original is not None}")
print(f"Small loaded: {small is not None}")

# Scaling factor to return small back to original size
s = original.shape[0] / small.shape[0]

# Scale up using both methods
upscaled_nn = zoom_image(small, s, method='nearest')
upscaled_bl = zoom_image(small, s, method='bilinear')

# Compute SSD
ssd_nn = compute_normalized_ssd(original, upscaled_nn)
ssd_bl = compute_normalized_ssd(original, upscaled_bl)

print(f"Normalized SSD (Nearest Neighbor): {ssd_nn:.2f}")
print(f"Normalized SSD (Bilinear): {ssd_bl:.2f}")

Original loaded: True
Small loaded: True


ValueError: setting an array element with a sequence.